### Занятие 2. Данные и управление

<sub>Слайд 56</sub>

**На прошлом занятии:** как исполняется программа, имена и ссылки на объекты,
`==` против `is`, изменяемые и неизменяемые объекты, числовые типы.

**Сегодня:**

1. Строки и форматирование.
2. Кортежи, списки, словари, множества.
3. Файлы и сериализация: `pickle`, `json`, `csv`.
4. Ветвления, циклы и протокол итерации.


**Подготовка**

Каталог для файлов, которые мы создадим в середине занятия.


In [ ]:
import pathlib, sys

DEMO = pathlib.Path('demo')
DEMO.mkdir(exist_ok=True)
print('Python', sys.version.split()[0], '| файлы будут здесь:', DEMO.resolve())

## Строки


### Коллекции, последовательности и строка

<sub>Слайд 57-59</sub>

* **Коллекция** — объект, который хранит ссылки на другие объекты.
* **Последовательность** — коллекция с порядком: `len(s)`, индекс `s[i]`, срез `s[a:b]`, проверка `in`.
* Неизменяемые последовательности: строка `str`, байтовая строка `bytes`, кортеж `tuple`.
* Изменяемые последовательности: список `list`, байтовый массив `bytearray`.
* **Строка** — упорядоченная последовательность 16-битных беззнаковых целых чисел, называемых элементами.
* Каждый элемент — символ Unicode, каждому элементу назначен индекс: `s[0]`, `s[1]`, …
* `len(s)` считает элементы, а не байты. Байты появляются при кодировании: `s.encode('utf-8')` → `bytes`, `b.decode('utf-8')` → `str`.


**Символы, байты и кодировка**

Одна и та же строка до и после кодирования в UTF-8, а затем попытка прочитать её не той кодировкой.


In [ ]:
for probe in ['abc', 'абв', '🐍']:
    encoded = probe.encode('utf-8')
    print(probe, '| символов:', len(probe), '| байт в utf-8:', len(encoded), encoded)

hello_bytes = 'Привет'.encode('utf-8')
print('обратно:', hello_bytes.decode('utf-8'))
print('первый элемент bytes - это число:', hello_bytes[0])
try:
    hello_bytes.decode('ascii')
except UnicodeDecodeError as e:
    print('UnicodeDecodeError:', e)

### Как создать строку: литералы, экранирование, raw и bytes

<sub>Слайд 60-63</sub>

* Способы создания: литералы, управляющие последовательности, неформатированные (raw) строки, функция `str`.
* Кавычки: `'allows embedded "double" quotes'`, `"allows embedded 'single' quotes"`.
* Тройные кавычки `'''...'''` и `"""..."""` — многострочный текст: сообщения об ошибках, фрагменты HTML, XML, JSON.
* Строки в тройных кавычках в определённых местах файла работают как строки документации.
* Управляющие последовательности: `'s\np\ta\x00m'` — перевод строки, табуляция, нулевой символ.
* Raw-строка без обработки экранирования: `r's\np\ta\x00m'` — пути Windows, регулярные выражения.
* Байтовая строка для двоичных данных и закодированного текста: `b's\np\ta\x00m'`.


**print против repr, экранирование и raw-строка**

`print` показывает строку как текст, `repr` — как её пришлось бы записать в коде.


In [ ]:
esc_demo = 's\np\ta\x00m'
print(esc_demo)
print('repr:', repr(esc_demo), '| len =', len(esc_demo))

path_plain = 'C:\new\table.txt'
path_raw = r'C:\new\table.txt'
print('без r:', repr(path_plain))
print('с r  :', repr(path_raw), '| равна экранированной:', path_raw == 'C:\\new\\table.txt')
print('bytes:', b's\np\ta\x00m', '| тип', type(b'').__name__)

### Функция str и операции над строками

<sub>Слайд 64-65</sub>

* `str(object=b'', encoding='utf-8', errors='strict')` — текстовое представление объекта.
* Если передан только объект, вызывается `type(object).__str__(object)`, а если такого метода нет — `repr(object)`.
* `__str__` и `__repr__` можно переопределить на уровне класса.
* В Python нет символа завершения строки: хранятся и длина, и текст.
* Длина `len('abc')`; склейка `'abc' + 'qwe'` и `'abc' 'qwe'`; повторение `'abc' * 7`.
* `'abc' + 7` — ошибка: не все операции строк с числами разрешены.
* Индексация и нарезание: `s[0]`, `s[-2]`, `s[1:3]`, `s[1:10:2]`.


**Индексы и срезы: что падает, а что нет**

Срезы со слайда, разворот строки отрицательным шагом и выход за границы.


In [ ]:
str_lit = 'abcdefghijklmno'
print(str_lit[0], str_lit[-2], str_lit[1:3], str_lit[1:10:2])
print('разворот        :', str_lit[::-1])
print('склейка половин :', str_lit[:4] + str_lit[4:] == str_lit)
print('срез за границей:', repr(str_lit[10:100]), repr(str_lit[100:200]))
try:
    str_lit[100]
except IndexError as e:
    print('IndexError:', e)
try:
    'abc' + 7
except TypeError as e:
    print('TypeError:', e)

### Коды символов, неизменяемость, обрезка пробелов

<sub>Слайд 66</sub>

* Коды символов: `ord('s')` → `115`, `chr(115)` → `'s'`, `chr(ord('s') + 1)` → `'t'`.
* Строки Python относятся к **неизменяемым** последовательностям: символы имеют позиционный порядок слева направо и не могут быть модифицированы на месте.
* `str_lit[0] = 'r'` — ошибка; новая строка вместо изменения: `'r' + str_lit[1:]`, `str_lit.replace('h', 'r')`.
* `strip()` — новая строка с обрезанными с обеих сторон пробелами.
* `rstrip()` — обрезка справа, `lstrip()` — обрезка слева.
* Все методы строк возвращают **новую** строку, исходная не меняется.


**Коды, неизменяемость и обрезка на практике**

Смотрим коды символов, пробуем изменить строку на месте и сверяем `id` до и после `replace`.


In [ ]:
print("ord('s') =", ord('s'), "| chr(115) =", chr(115), "| chr(ord('s') + 1) =", chr(ord('s') + 1))
print("ord('A') =", ord('A'), "| ord('a') =", ord('a'), "| 'Z' < 'a':", 'Z' < 'a')

str_lit = 'hello'
print('id исходной строки:', id(str_lit))
try:
    str_lit[0] = 'r'
except TypeError as e:
    print('TypeError:', e)
replaced = str_lit.replace('h', 'r')
print('replace дал:', replaced, '| id нового объекта:', id(replaced))
print('исходная осталась:', str_lit, '| срез и склейка:', 'r' + str_lit[1:])

raw_line = '   Иванов И.И.  \t\n'
print(repr(raw_line.strip()), repr(raw_line.lstrip()), repr(raw_line.rstrip()))
print('обрезка по набору символов:', repr('xyzabcyx'.strip('xyz')))

**Склейка в цикле против join**

Собираем одну и ту же длинную строку двумя способами и замеряем время.


In [ ]:
import time
parts_demo = ['слово'] * 20000
t0 = time.perf_counter()
acc = ''
for w in parts_demo:
    acc = acc + w
t_concat = time.perf_counter() - t0

t0 = time.perf_counter()
joined = ''.join(parts_demo)
t_join = time.perf_counter() - t0
print('результаты равны:', acc == joined)
print('плюс в цикле:', round(t_concat, 4), 'с')
print('join        :', round(t_join, 4), 'с')

## Форматирование и методы строк


### Форматирование строк: %, str.format и f-строки

<sub>Слайд 67-72</sub>

* Через `%`, по модели `printf` из C, применялся в Python 2: `'... %s ...' % (value1, value2)`.
* Цели преобразования: `%s` — в строку, `%d` и `%i` — в целое, `%f` — в число с плавающей точкой.
* Общий вид: `%[key][flags][width][.precision][length_type][conversion_type]`; флаги `%05d`, `%-5d`, `% 5d`, `%+5d`; точность `'%.2f' % 0.333333`, `'%.5s' % 'formatting'`.
* Через метод `str.format`: `'... {} ...'.format(value)` — позиционные `{}`, нумерованные `{0} {2}`, именованные `{first_name}`, распаковка словаря `.format(**person)`.
* Формат после двоеточия: `{weight:.2f}`, `{weight:07.2f}`; выравнивание `{0:>5d}`, `{0:<5d}`, `{0:^5d}`, по умолчанию `{0:5d}`.
* f-строка: `f'... {value} ...'` — значение или выражение прямо в поле замены, `{person['age']}`, спецификация та же: `{person['weight']:07.2f}`.


**Три способа — один результат**

Одна строка отчёта, собранная тремя способами, а затем спецификации формата по частям.


In [ ]:
epoch_no, metric_name, metric_value = 7, 'accuracy', 0.876543
line_percent = 'epoch %2d: %-10s = %.4f' % (epoch_no, metric_name, metric_value)
line_format = 'epoch {:2d}: {:<10} = {:.4f}'.format(epoch_no, metric_name, metric_value)
line_fstring = f'epoch {epoch_no:2d}: {metric_name:<10} = {metric_value:.4f}'
print(line_fstring, '| все три равны:', line_percent == line_format == line_fstring)

weight_demo = 72.5
print(f'[{weight_demo:07.2f}] [{weight_demo:>8}] [{weight_demo:<8}] [{weight_demo:^8}]')
print(f'{1234567:,}', '|', '{:.1%}'.format(0.8734), '|', '%+5d|' % 1)
print(f'{weight_demo = }')

### Методы строк: replace, split, join, find

<sub>Слайд 73-74</sub>

* `replace` принимает исходную подстроку и строку-замену и выполняет глобальный поиск и замену: `'axa byb cyc'.replace('x', 'a')`.
* Третий аргумент ограничивает число замен: `str_obj.replace('y', 'b', 1)`.
* `split(sep)` разбивает строку на список подстрок по строке-разделителю: `str_obj.split(' ')`.
* `join` объединяет строки из списка или другого итерируемого объекта, вставляя между ними разделитель: `' '.join(str_list)`.
* `find` возвращает смещение, по которому обнаружена подстрока, или `-1`, если её нет: `str_obj.find('a')`.
* `index` делает то же, но при отсутствии подстроки возбуждает `ValueError`.


**split, join, find и index**

Строка с лишними пробелами разбивается двумя способами, затем собираем её обратно и ищем подстроку.


In [ ]:
spaced = '  раз  два   три '
print("split(' '):", spaced.split(' '))
print('split()   :', spaced.split())
print('join      :', '-'.join(spaced.split()))

str_obj = 'axa byb cyc'
print('replace:', str_obj.replace('y', 'b'), '|', str_obj.replace('y', 'b', 1))
print("find('b'):", str_obj.find('b'), "| find('zzz'):", str_obj.find('zzz'))
try:
    str_obj.index('zzz')
except ValueError as e:
    print('ValueError:', e)

## Кортежи, range и списки


### Кортеж и range

<sub>Слайд 75-76</sub>

* **Кортеж** — неизменяемая упорядоченная последовательность произвольных объектов, доступ по смещению.
* Внутри — массив **ссылок** на объекты фиксированной длины.
* Создание: `()`, `tuple()`, `(1, 2, 'var')`, `1, 2, 'var'`, вложенный `(1, 2, ('var', 3))`.
* Кортеж из одного элемента — `(1,)`; а `(1)` — просто число в скобках: кортеж делает **запятая**.
* Операции: `t[i][j]`, `t[i:j]`, `t.index(1)`, `t.count(1)`; `+`, `*` и нарезание возвращают **новые** кортежи.
* Неизменяемость применяется только к **верхнему уровню**: список внутри кортежа меняется.
* Распаковка: `a, b = b, a`, `first, *rest = t`.
* **`range(start, stop, step)`** — неизменяемая последовательность чисел; элементы не хранятся, а вычисляются.
* Сравнивается последовательность: `range(0, 5) == range(0, 6)` → `False`.


**Запятая, вложенный список, распаковка и размер range**

Что делает кортеж кортежем, где кончается его неизменяемость и сколько памяти стоит `range`.


In [ ]:
print('(1) это', type((1)), 'а (1,) это', type((1,)))
tup_with_list = (1, 2, [3, 4])
tup_with_list[2].append(5)
print('список внутри изменился:', tup_with_list)
try:
    tup_with_list[0] = 100
except TypeError as e:
    print('TypeError:', e)
first, *rest = tup_with_list
print('first =', first, '| rest =', rest)
import sys
print('range(10**9):', sys.getsizeof(range(10**9)), 'байт |',
      'list(range(10**6)):', sys.getsizeof(list(range(10**6))), 'байт')
print('range(0, 5) == range(0, 6):', range(0, 5) == range(0, 6),
      '| 999 in range(10**9):', 999 in range(10**9))

### Список

<sub>Слайд 77-78</sub>

* **Изменяемая** упорядоченная последовательность произвольных объектов, доступ по смещению.
* По сути — **массив ссылок** на объекты, выделенный с запасом.
* Создание: `L = []`, `L = [1, 'abc', [1, 2, 3], {}]`, `L = list('example')`.
* Доступ и нарезание: `L[i]`, `L[i][j]`, `L[i:j]`, `L[i:j:k]`, `L[:-1]`; присваивание `L[i] = 'test'`.
* Срез возвращает **новый** список; копируются ссылки, а не вложенные объекты.
* Методы: `append`, `extend`, `insert`, `pop`, `remove`, `index`, `count`.
* `L.sort()` меняет список на месте и возвращает `None`; `sorted(L)` возвращает новый список.
* Оба принимают `key=` (функция для сравнения) и `reverse=`.


**Срезы, методы и сортировка**

Основные операции со списком и разница между `sort` и `sorted`.


In [ ]:
letters = list('example')
print(letters[1:4], letters[::2], letters[:-1])
letters.append('!'); letters.insert(0, '>'); letters.pop(1)
print(letters, '| count e:', letters.count('e'))

scores = [('resnet', 0.91), ('vit', 0.94), ('mlp', 0.78)]
def by_metric(pair):
    return pair[1]

print(sorted(scores, key=by_metric, reverse=True))
print('scores.sort() вернул:', scores.sort())
print('а scores теперь:', scores)

## Словари и множества


### Словарь: зачем он нужен и как устроен

<sub>Слайд 79-80, 84</sub>

* **Изменяемая** коллекция объектов любого типа с доступом **по ключу**, а не по смещению.
* Таблица ссылок на объекты; другое название — **ассоциативный массив**. Допускает произвольную глубину вложения.
* Внутри реализован как **хеш-таблица**: поиск, вставка и удаление в среднем за одно обращение.
* Ключом может быть любой **хешируемый** объект: строка, число, кортеж из неизменяемых.
* Создание: `{}`, `dict()`, литерал `{'key1': 'value1'}`, `dict(key3=3, key4=4)`,
  `dict([('key3', 3), ('key4', 4)])`, `dict(zip(keys, values))`, `dict.fromkeys(['key3', 'key4'], 0)`.
* **Разреженные структуры**: список с элементами на позициях 0, 20 и 99 требует памяти под все сто,
  а словарь хранит только три записи.
* Многомерный массив: ключ — **кортеж индексов**, `tensor[(0, 1, 1)] = 11`.


**Создание словаря, неизменяемый ключ и разреженная матрица**

Один и тот же словарь разными способами, попытка сделать ключом список и матрица 1000×1000 из трёх элементов.


In [ ]:
import sys
feature_names, feature_values = ['key3', 'key4'], [3, 4]
variants = [{'key3': 3, 'key4': 4}, dict(key3=3, key4=4),
            dict([('key3', 3), ('key4', 4)]), dict(zip(feature_names, feature_values))]
for v in variants:
    print(v, '| равен первому:', v == variants[0])
print('fromkeys:', dict.fromkeys(['key3', 'key4'], 0))
try:
    bad_dict = {[1, 2]: 'list key'}
except TypeError as e:
    print('TypeError:', e)
sparse_matrix = {(0, 1): 11, (2, 999): 21, (500, 500): 7}
print('элемент (2, 999):', sparse_matrix[2, 999], '| элемент (3, 3):', sparse_matrix.get((3, 3), 0))
print('ненулевых:', len(sparse_matrix), 'из', 1000 * 1000)
print('словарь:', sys.getsizeof(sparse_matrix), 'байт; плотный список:',
      sys.getsizeof([0] * 10**6), 'байт')

### Ключи, представления и отсутствующий ключ

<sub>Слайд 81-83</sub>

* `keys()`, `values()`, `items()` возвращают **объекты представлений**.
* Представления **итерируемы**: выдают элементы по одному, а не загружают весь список в память.
* Представления **живые**: видят изменения, произошедшие в словаре после их создания.
* Поддерживают операции над множествами; индексации и метода `sort`, как у списка, у них нет.
* Сортировка: `sorted(d)` — отсортированный список ключей.
* Отсутствующий ключ даёт `KeyError`. Способы обработки:
  * `if 'key3' in d8: ... else: ...`
  * `try: ... except KeyError: ...`
  * `d8.get('key3', 0)`
* Ещё есть `d.setdefault(key, [])` и `collections.defaultdict`.


**Живое представление и три способа обойти KeyError**

Сохраним `items()` и изменим словарь, а затем посчитаем частоты слов тремя способами.


In [ ]:
from collections import defaultdict
config = {'lr': 0.1, 'epochs': 10}
config_items = config.items()
config['batch'] = 32
config['lr'] = 0.01
print('представление после изменений:', config_items)
print('общие ключи:', config.keys() & {'lr', 'dropout'})
tokens = 'the cat and the dog and the bird'.split()
counts_get, groups_setdefault, counts_default = {}, {}, defaultdict(int)
for word in tokens:
    counts_get[word] = counts_get.get(word, 0) + 1
    groups_setdefault.setdefault(len(word), []).append(word)
    counts_default[word] += 1
print('через get:        ', counts_get)
print('через setdefault: ', groups_setdefault)
print('defaultdict дал то же:', dict(counts_default) == counts_get)

### Множество и frozenset

<sub>Слайд 85-86</sub>

* **Неупорядоченная** коллекция **уникальных** и неизменяемых объектов с операциями теории множеств.
* Элемент встречается во множестве однажды, сколько бы раз его ни добавляли.
* Элемент обязан обладать свойством **хешируемости**.
* Множества итерируемы, растут и уменьшаются по требованию, содержат объекты разных типов.
* Не последовательность и не отображение — **отдельная категория** типов.
* Операции: `|` объединение, `&` пересечение, `-` разность, `^` симметрическая разность, `<=` подмножество.
* Само множество изменяемо и потому не вкладывается в другое множество; для этого есть **`frozenset`**.
* Применения: фильтрация дубликатов (`set(L)`, порядок теряется), различия между коллекциями, быстрая проверка `x in s`.
* Пустое множество — `set()`, а `{}` — пустой словарь.


**Операции над множествами и скорость проверки in**

Две выборки пользователей, `frozenset` в роли ключа и замер `in` на ста тысячах элементов.


In [ ]:
import time
train_users = {'ann', 'bob', 'kim', 'lee'}
test_users = {'kim', 'max', 'zoe'}
print('утечка (&):', train_users & test_users)
print('только в train (-):', sorted(train_users - test_users))
print('в одной из выборок (^):', sorted(train_users ^ test_users))
edge_weights = {frozenset({'A', 'B'}): 5}
print('ребро B-A:', edge_weights[frozenset({'B', 'A'})])

numbers_list = list(range(100000))
numbers_set = set(numbers_list)
for name, container in [('list', numbers_list), ('set', numbers_set)]:
    start = time.perf_counter()
    for _ in range(200):
        found = -1 in container
    print(f'{name}: {(time.perf_counter() - start) / 200 * 1e6:10.2f} мкс на проверку')

## Файлы и сериализация


### Файловый объект, with и режимы открытия

<sub>Слайд 87-90</sub>

* Встроенная функция `open` создаёт **файловый объект** — ссылку на файл; у него только методы для типичных задач.
* Данные, считываемые из текстового файла, всегда возвращаются в виде строки `str`.
* Три способа закрыть файл:
  * `file1 = open(path, mode='w', encoding='utf8')` … `file1.close()`;
  * `try: ... finally: file3.close()`;
  * диспетчер контекста `with open(path, mode='w') as file2:` — закрывает сам.

| режим | смысл |
|---|---|
| `'r'` | чтение (по умолчанию, синоним `'rt'`) |
| `'w'` | запись, файл **сначала обнуляется** |
| `'x'` | создание; ошибка, если файл уже есть |
| `'a'` | дозапись в конец |
| `'b'` | двоичный режим |
| `'t'` | текстовый режим (по умолчанию) |
| `'+'` | чтение и запись: `'w+'` перезаписывает, `'r+'` — нет |

* **Текстовый** файл отдаёт `str`, сам кодирует и декодирует Unicode и переводит концы строк; **двоичный** отдаёт `bytes` без изменений.


**Запись и построчное чтение через with**

Пишем журнал обучения и читаем его обратно по строкам.


In [ ]:
log_path = DEMO / 'train_log.txt'
with open(log_path, 'w', encoding='utf-8') as log_file:
    for epoch, loss in [(1, 0.92), (2, 0.61), (3, 0.47)]:
        log_file.write(f'эпоха {epoch}: loss={loss}\n')
print('закрыт после with:', log_file.closed)

with open(log_path, encoding='utf-8') as log_file:
    for line in log_file:
        print(repr(line))

### Объект в файл и обратно: eval, pickle, JSON, CSV

<sub>Слайд 91-95</sub>

* Чтобы сохранить объект в текстовый файл, его нужно представить в виде строки: `d1_file.write(str(d1))`.
* Обратно — `eval(d1_file.read())`: строка **исполняется как код Python**, использовать опасно.
* **`pickle`** сохраняет почти любой объект Python напрямую, не требуя преобразований в строку и обратно.
* `pickle` выполняет **сериализацию** — преобразование объекта в строку байтов и обратно.
* `pickle.dump(d1, f)` с режимом `'wb'`, `pickle.load(f)` с режимом `'rb'`.
* **`json`** — сериализация специфической категории объектов; синтаксически близок к словарям и спискам Python.
* `json.dumps(d1)`, `json.dump(d1, fp, indent=4)`, `json.load(f)`.
* JSON знает только `dict`, `list`, `str`, числа, `true/false/null`: **кортеж станет списком**, **ключ-число — строкой**.
* **`csv`** — табличные данные: `csv.writer(f, delimiter=',', quotechar='"', quoting=csv.QUOTE_MINIMAL)`, `writerow`; `csv.reader(f)` отдаёт списки строк.


**Один объект через pickle и через json**

Сохраняем одинаковый словарь двумя способами и сравниваем, что вернулось.


In [ ]:
import pickle, json

experiment = {'shape': (3, 4), 1: 'epoch one', 'lr': 0.01}
with open(DEMO / 'experiment.pkl', 'wb') as pkl_file:
    pickle.dump(experiment, pkl_file)
with open(DEMO / 'experiment.pkl', 'rb') as pkl_file:
    from_pickle = pickle.load(pkl_file)
with open(DEMO / 'experiment.json', 'w', encoding='utf-8') as json_file:
    json.dump(experiment, json_file)
print('json на диске:', (DEMO / 'experiment.json').read_text(encoding='utf-8'))
with open(DEMO / 'experiment.json', encoding='utf-8') as json_file:
    from_json = json.load(json_file)
print('pickle:', from_pickle, '| равен исходному:', from_pickle == experiment)
print('json:  ', from_json, '| равен исходному:', from_json == experiment)

**csv.writer и csv.reader**

Таблица сотрудников: запись и чтение, причём в одном значении спрятана запятая.


In [ ]:
import csv

employees_path = DEMO / 'employees.csv'
with open(employees_path, 'w', encoding='utf-8', newline='') as csv_file:
    employee_writer = csv.writer(csv_file, delimiter=',', quotechar='"', quoting=csv.QUOTE_MINIMAL)
    employee_writer.writerow(['first_name', 'last_name', 'age'])
    employee_writer.writerow(['ivan', 'ivanov', 27])
    employee_writer.writerow(['petr', 'petrov, jr', 37])
print(employees_path.read_text(encoding='utf-8'))

with open(employees_path, encoding='utf-8', newline='') as csv_file:
    for row in csv.reader(csv_file):
        print(row)

## Условный оператор


### if / elif / else, тернарный оператор и словарь вместо switch

<sub>Слайд 96-99</sub>

```python
if condition1:                 check = 'x'
    statements1                dict_as_switch = {'x': 1, 'y': 2, 'z': 3}
elif condition2:               print(dict_as_switch[check])
    statements2
else:                          value = 1 if x == 3 else 0
    statements3
```

* `if` — **составной оператор**: в него встроены другие операторы, выделенные отступом.
* `elif` («else if») — сколько угодно, `else` — не больше одного и в конце.
* Выполняется блок **первой** истинной проверки, иначе блок `else`.
* **Тернарный оператор** — выражение со значением: `1 if x == 3 else 0`.
* Условием может быть любой объект: ложны `0`, `0.0`, `''`, `[]`, `{}`, `set()`, `None`.
* Оператора `switch`/`case` в Python 3.9 нет. Множественное ветвление — цепочка `if/elif`, **индексирование по словарю** или поиск по списку.
* Словарь удобно **менять динамически** во время работы программы; его значениями могут быть **функции**.


**Словарь функций вместо цепочки elif**

Калькулятор: действие выбирается по ключу, неизвестное обрабатывается через `.get`.


In [ ]:
def add(a, b):
    return a + b

def sub(a, b):
    return a - b

operations = {'+': add, '-': sub}
operations['**'] = pow                    # вариант добавлен во время работы

for op in ('+', '-', '**', '%'):
    action = operations.get(op)
    if action is None:
        print(op, '-> неизвестная операция')
    else:
        print('7', op, '2 =', action(7, 2))

## Циклы while и for


### while и for: break, continue, pass, else

<sub>Слайд 100-103</sub>

```python
while проверка:            for цель in объект:
    операторы                  операторы
    if ...: break              if ...: break
    if ...: continue           if ...: continue
else:                      else:
    операторы  # если не было break
```

* `break` — немедленный выход из цикла, `else` при этом пропускается.
* `continue` — немедленный переход к проверке в начале цикла; похож на `goto`, поэтому к нему относятся все предупреждения про читабельность.
* `pass` — заполнитель там, где синтаксис требует оператора, а делать нечего.
* `else` выполняется, если цикл завершился **без `break`** (в том числе если тело не выполнилось ни разу). Часто устраняет потребность во флаге поиска.
* `for` по очереди присваивает цели элементы объекта, как курсор, идущий по последовательности.
* После цикла цель по-прежнему ссылается на последний посещённый элемент.


**Поиск делителя: флаг против while/else**

Одна и та же задача двумя способами: найти наименьший делитель числа.


In [ ]:
number, divisor, found = 91, 2, False     # версия с флагом
while divisor * divisor <= number:
    if number % divisor == 0:
        found = True
        break
    divisor += 1
print('флаг:', divisor if found else 'простое')
for number in (91, 97):                   # та же задача через while/else
    divisor = 2
    while divisor * divisor <= number:
        if number % divisor == 0:
            print(number, '-> делитель', divisor)
            break
        divisor += 1
    else:
        print(number, '-> простое')

### for по любому объекту: range, zip, enumerate

<sub>Слайд 104-105</sub>

* `for` работает не только с последовательностями: со словарями, множествами, файлами (построчно).
* Если элементы — кортежи, цель тоже может быть кортежем: `for (a, b) in [(1, 2), (3, 4)]:` — обычное распаковывающее присваивание.
* `range(0, 20, 2)` — числа по запросу, без создания списка.
* `zip(L1, L2, L3)` — параллельный обход нескольких коллекций, **обрезает по самой короткой**.
* `enumerate('hello world')` — пары `(индекс, элемент)`, есть параметр `start`.
* `for k, v in d.items():` — обход словаря сразу с ключами и значениями.
* Коллекцию, по которой идёт цикл, изменять нельзя.


**zip, enumerate и удаление во время прохода**

Параллельный обход с нумерацией, а затем попытка убрать из списка чётные числа прямо в цикле.


In [ ]:
names = ['Аня', 'Борис', 'Вера']
scores = [91, 78, 85, 60]                 # одна оценка лишняя
for place, (name, score) in enumerate(zip(names, scores), start=1):
    print(place, name, score)
print('zip молча отбросил оценок:', len(scores) - len(names))

values = [2, 4, 5, 6, 8, 9]               # хотим убрать все чётные
for v in values:
    if v % 2 == 0:
        values.remove(v)
print('удаляли на ходу:', values)
odd_values = []                           # правильный способ: строим новый список
for v in [2, 4, 5, 6, 8, 9]:
    if v % 2 != 0:
        odd_values.append(v)
print('новый список:   ', odd_values)

## Итераторы


### Протокол итерации: iter, next, StopIteration

<sub>Слайд 106-108</sub>

```python
L1 = [1, 2, 3, 4, 5]      # 1. итерируемый объект
I1 = iter(L1)             # 2. итератор из него
print(next(I1))           # 3. следующий элемент
print(I1.__next__())      #    то же самое
```

* Циклы работают не только с последовательностями, а с любым **итерируемым объектом**; то же верно для всех инструментов, просматривающих объекты слева направо: `map`, `filter`, `in`, `sum`, `sorted`.
* `map(f, данные)` применяет `f` к каждому элементу; `filter(f, данные)` оставляет элементы, для которых `f` истинна.
* **Итерируемый объект** — поддерживает вызов `iter()`; **итератор** — то, что `iter()` вернул, поддерживает `next()` и помнит позицию.
* Когда элементы кончились, `next()` возбуждает `StopIteration`.
* Итерируемыми могут быть и физические, и **виртуальные** (вычисляемые по запросу) последовательности.
* Объект, поддерживающий много открытых итераций, сам итератором **не является** (список).
* Файл, `map`, `filter`, `zip`, `enumerate` — итераторы: одноразовые и ленивые.


**Цикл for, написанный вручную**

Сначала вызываем `iter` и `next` руками, потом собираем из них обычный `for`.


In [ ]:
letters = iter('abc')                     # итератор поверх строки
print(next(letters), letters.__next__())
try:
    next(letters)
    next(letters)
except StopIteration:
    print('StopIteration: элементы кончились')

iterator = iter([10, 20, 30])             # ровно это делает for x in [10, 20, 30]
while True:
    try:
        x = next(iterator)
    except StopIteration:
        break
    print('тело цикла, x =', x)

**Итератор одноразовый, список — нет**

`map(square, [1, 2, 3])` применяет `square` к каждому элементу. Посмотрим, когда он это делает и что будет при втором проходе.


In [ ]:
def square(x):
    print('  считаю', x)
    return x * x

squares = map(square, [1, 2, 3])
print('map создан, ни одного вызова square:', squares)
print('первый проход: ', list(squares))
print('второй проход: ', list(squares))

numbers = [1, 2, 3]
print('список, проход 1:', sum(numbers))
print('список, проход 2:', sum(numbers))
print('iter(numbers) is numbers:', iter(numbers) is numbers)

### Итоги занятия

<sub>Слайд 56-108</sub>

* Коллекции хранят **ссылки** на объекты: отсюда копирование, изменяемость и поведение вложенных структур.
* Строки и кортежи неизменяемы, списки, словари и множества изменяемы.
* Словарь и множество — поиск по хешу за постоянное время; ключи должны быть хешируемыми.
* `json` и `csv` — для обмена данными, `pickle` — только для своих объектов Python.
* Ложны `0`, пустые коллекции и `None`; вместо длинной цепочки `elif` — словарь, в том числе словарь функций.
* Не ходите по индексам: `zip`, `enumerate`, `items()`; не меняйте коллекцию, по которой идёте.
* `for` = `iter()` + `next()` до `StopIteration`; итераторы ленивы и одноразовы.

**Дальше:** функции, области видимости, замыкания, декораторы, включения, генераторы, ООП.
